In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import trimesh
from globe3d import (
    generate_sphere_points_fibonacci,
    load_netcdf_grid,
    calculate_displacement_scale,
    displace_vertices,
    assign_vertex_colors,
    combine_subtractive_globes,
    split_mesh_hemispheres,
    write_obj_with_vertex_colors
)

In [ ]:
# Model parameters from tomo_globe.ipynb
outer_points = 1000000 # number of vertices in the outer shell
inner_points = 20000 # number of vertices in the inner shell
vert_exagg = 30 # factor by which to exaggerate topography
earth_radius_km = 6371.0
model_radius_mm = 40.0 # 80mm diameter globe
inner_scale = 0.8 # fraction of the size of the inner void relative to the outer

print("Generating base spheres...")
outer_vertices, outer_faces = generate_sphere_points_fibonacci(outer_points, model_radius_mm)
inner_vertices, inner_faces = generate_sphere_points_fibonacci(inner_points, model_radius_mm * inner_scale)

In [ ]:
dem_grid = r"F:\no_backup\gmt_grids\ETOPO_2022_v1_60s_N90W180_surface.nc"
colour_grid = r"D:\OneDrive\python\3d_globes\tomo_cross_section\s40_depth_slice_2850.grd"

print("Loading ETOPO topography grid...")
topo_lats, topo_lons, topo_grid = load_netcdf_grid(dem_grid, lat_var='lat', lon_var='lon', data_var='z')

print("Loading colour grid...")
c_lats, c_lons, c_grid = load_netcdf_grid(colour_grid, lat_var='y', lon_var='x', data_var='z')


In [ ]:
print("Displacing outer vertices...")
scale = calculate_displacement_scale(model_radius_mm, earth_radius_km, vertical_exagg=vert_exagg)
outer_vertices = displace_vertices(outer_vertices, topo_lats, topo_lons, topo_grid, scale, show_progress=True)

In [ ]:
print("Combining subtractive globes (hollowing)...")
combined_vertices, combined_faces = combine_subtractive_globes(outer_vertices, outer_faces, inner_vertices, inner_faces)
print(f"Combined mesh has {len(combined_vertices)} vertices and {len(combined_faces)} faces.")

In [ ]:
print("Creating mesh and splitting...")
mesh = trimesh.Trimesh(vertices=combined_vertices, faces=combined_faces)

print("Splitting mesh into hemispheres...")
top_half, bottom_half = split_mesh_hemispheres(mesh)

In [ ]:
# --- Colouring Options ---
# Option 1: Colour according to boundaries (current workflow)
cmap_bounds = mcolors.ListedColormap(['blue', 'white', 'red'])
norm = mcolors.BoundaryNorm([-10, -0.5, 0.5, 10], cmap_bounds.N)
cmap = cmap_bounds
colour_kwargs = {'norm': norm}

# Option 2: Colour using a continuous matplotlib cmap
# cmap = 'RdBu_r'
# colour_kwargs = {'vmin': -1, 'vmax': 1,}

# Option 3: Colour using a discretised version of a matplotlib cmap
cmap = plt.get_cmap('RdBu_r', 7) # 10 discrete bins
colour_kwargs = {'vmin': -2, 'vmax': 2,}

In [ ]:
# --- Preview 2D Colour Map ---
plt.figure(figsize=(10, 5))
lon_grid, lat_grid = np.meshgrid(c_lons, c_lats)
plt.pcolormesh(lon_grid, lat_grid, c_grid, cmap=cmap, **colour_kwargs, shading='auto')
plt.colorbar(label='Value')
plt.title('2D Preview of Colour Grid')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [ ]:
if top_half and bottom_half:
    print("Assigning colors to split halves...")
    top_colors = assign_vertex_colors(top_half.vertices, c_lats, c_lons, c_grid, colormap=cmap, **colour_kwargs)
    bottom_colors = assign_vertex_colors(bottom_half.vertices, c_lats, c_lons, c_grid, colormap=cmap, **colour_kwargs)
    
    print("Exporting...")
    write_obj_with_vertex_colors('tomo_globe_top.obj', top_half.vertices, top_half.faces, top_colors)
    write_obj_with_vertex_colors('tomo_globe_bottom.obj', bottom_half.vertices, bottom_half.faces, bottom_colors)
    print("Exported split globes successfully.")
else:
    print("Mesh splitting failed.")